# Domain 5: Model Selection and Optimization
**Exam**: Claude Certified Developer – Foundations (CCDV-F)
**Weighting**: 16.8% of exam
**Depth**: deep

## Domain Overview

This domain covers LLM fundamentals (tokens, context windows, sampling, non-determinism),
technical fundamentals (SDKs wrapping REST/websockets), model selection tradeoffs (Opus/Sonnet/
Haiku, adaptive thinking, breaking changes across releases), and cost/token management (tracking,
modeling, caching).

**Why it matters**: at 16.8% weight, developers must be fluent in the fundamentals that drive every
cost, latency, and quality decision in a Claude application.

**Key concepts to know**: token/context fundamentals, model tier tradeoffs, thinking modes/effort
levels, and prompt caching for cost optimization.

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### LLM Fundamentals: tokens, context windows, sampling, non-determinism

Tokens are the unit of both input/output length and cost. Context windows bound how much
conversation/tool history can be included. Sampling (temperature, top-p) introduces controlled
randomness — meaning identical prompts can produce different outputs (non-determinism), which
matters for testing and reproducibility.

In [ ]:
def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)  # rough heuristic: ~4 chars/token

print(estimate_tokens("This is a rough token estimate for planning purposes."))
print("Non-determinism note: identical prompts CAN produce different outputs due to sampling; design tests accordingly (e.g., check structure, not exact text).")

### Model options: fast mode, extended thinking, adaptive thinking, effort levels

Fast mode favors low latency for simple tasks. Extended/adaptive thinking lets the model
allocate more internal reasoning for harder problems, improving quality at the cost of latency/tokens
— adaptive thinking adjusts this automatically based on perceived task difficulty rather than a fixed
setting.

In [ ]:
def choose_thinking_mode(task_complexity: str, latency_sensitive: bool) -> str:
    if latency_sensitive and task_complexity == "low":
        return "fast mode"
    if task_complexity == "high":
        return "extended/adaptive thinking - allow more reasoning budget"
    return "standard mode"

print(choose_thinking_mode("low", True))
print(choose_thinking_mode("high", False))

### Model Selection and Tradeoffs across Opus/Sonnet/Haiku and releases

Opus favors capability, Sonnet balances cost/capability, Haiku favors speed/cost at high
volume. Model releases can introduce breaking behavior changes (a prompt tuned for one version may
behave differently on the next) — production systems should evaluate against a fixed eval suite
before adopting a new model version.

In [ ]:
def model_for_task(complexity: str, volume: str) -> str:
    if volume == "high" and complexity in {"low", "medium"}:
        return "haiku"
    if complexity == "high":
        return "opus"
    return "sonnet"

print(model_for_task("low", "high"))
print(model_for_task("high", "low"))

print("Before adopting a new model version in production: re-run the eval suite to catch breaking behavior changes.")

### Cost and Token Management: tracking, modeling, and prompt caching

Cost modeling multiplies token volume by per-model rates across expected request types.
Prompt caching (reusing a stable prefix) is the highest-leverage lever for cutting cost/latency on
repeated system prompts or large static context, provided the static content is ordered first in the
request.

In [ ]:
def monthly_cost_estimate(requests_per_month: int, avg_input_tokens: int, avg_output_tokens: int,
                          input_rate_per_1k: float, output_rate_per_1k: float) -> float:
    input_cost = requests_per_month * (avg_input_tokens / 1000) * input_rate_per_1k
    output_cost = requests_per_month * (avg_output_tokens / 1000) * output_rate_per_1k
    return round(input_cost + output_cost, 2)

print(monthly_cost_estimate(500_000, 800, 200, input_rate_per_1k=0.003, output_rate_per_1k=0.015))

## Mini Hands-On Project

**Covers**: all Domain 5 sub-objectives.

**Scenario**: Choose a model tier and thinking mode, and estimate monthly cost, for a support-ticket
triage system processing 300,000 tickets/month.

### Step 1: Estimate tokens and choose the thinking mode

In [ ]:
print(estimate_tokens("A typical support ticket description is about this long, maybe 40-60 words."))
print(choose_thinking_mode("low", True))

### Step 2: Select the model tier

In [ ]:
print(model_for_task("low", "high"))

### Step 3: Estimate monthly cost with and without caching benefit

In [ ]:
cost_no_caching = monthly_cost_estimate(300_000, 600, 150, 0.0008, 0.004)  # haiku-tier rates, illustrative
print(cost_no_caching)

## Lab Exercises

### Exercise 1: Token estimation and budget check

**Objective**: Estimate tokens for 3 prompts and check against a 4000-token budget.

**Produce**: 3 estimates and a budget pass/fail.

In [ ]:
prompts = [
    "Short prompt: classify this ticket.",
    "Medium prompt: " + "context sentence. " * 50,
    "Long prompt: " + "context sentence. " * 500,
]
# TODO: use estimate_tokens and check against a 4000-token budget for each
for p in prompts:
    pass

# Expected output: Correct pass/fail results, with the long prompt likely exceeding the budget.

In [ ]:
# STRETCH GOAL: propose a trimming strategy for the prompt(s) that exceed budget.
trimming_strategy = "..."  # TODO

# Expected output: A concrete trimming strategy (e.g., summarize older context, drop low-relevance sentences).

### Exercise 2: Thinking mode selection for 4 tasks

**Objective**: Choose the thinking mode for 4 new tasks.

**Produce**: 4 mode choices with reasoning.

In [ ]:
tasks = [
    ("Real-time typo correction suggestion", "low", True),
    ("Complex multi-step tax calculation", "high", False),
    ("Simple sentiment tag", "low", True),
    ("Ambiguous legal contract interpretation", "high", True),
]
# TODO: call choose_thinking_mode for each
for desc, complexity, latency_sensitive in tasks:
    pass

# Expected output: Correct mode chosen per task, noting the 4th task is both high-complexity AND latency-sensitive - a genuine tradeoff to call out.

In [ ]:
# STRETCH GOAL: for the 4th (conflicting) task, propose a hybrid approach (e.g., fast initial
# pass + escalate to extended thinking only if confidence is low).
hybrid_approach = "..."  # TODO

# Expected output: A sensible hybrid/tiered approach resolving the complexity-vs-latency conflict.

### Exercise 3: Model tier migration risk check

**Objective**: Design a pre-adoption eval checklist for migrating to a new model version.

**Produce**: A checklist of at least 4 items.

In [ ]:
# TODO: list at least 4 things to check before adopting a new Claude model version in production
migration_checklist = []  # TODO
print(migration_checklist)

# Expected output: A checklist covering eval suite re-run, cost/latency comparison, prompt behavior spot-checks, and a rollback plan.

In [ ]:
# STRETCH GOAL: propose a staged rollout percentage plan (e.g., 5% -> 25% -> 100% traffic) with
# a go/no-go metric at each stage.
staged_rollout = {}  # TODO
print(staged_rollout)

# Expected output: A staged rollout plan with a clear metric gate at each percentage increase.

### Exercise 4: Cost model sensitivity analysis

**Objective**: Compare monthly cost across 3 model tiers for the same workload.

**Produce**: 3 cost estimates and the resulting savings if migrating from the most to least expensive.

In [ ]:
rates = {"haiku": (0.0008, 0.004), "sonnet": (0.003, 0.015), "opus": (0.015, 0.075)}
# TODO: call monthly_cost_estimate for each tier at 300,000 requests/month, 600 input / 150 output tokens
costs = {}
for tier, (in_rate, out_rate) in rates.items():
    pass
print(costs)

# Expected output: Three cost figures with opus substantially higher than haiku, and a correctly computed savings figure.

In [ ]:
# STRETCH GOAL: estimate the cost impact of adding prompt caching that reuses an 800-token
# static prefix on 80% of requests (assume cached tokens cost ~10% of normal input rate).
cached_prefix_tokens = 800
cache_hit_rate = 0.8
cache_discount = 0.10
# TODO: compute the adjusted monthly cost for the "sonnet" tier with this caching assumption

# Expected output: A meaningfully lower adjusted cost for the sonnet tier reflecting the caching discount.

## Self-Assessment: Model Selection and Optimization

Answer each question, then check the Answers cell below.


**Q1.** Why does non-determinism in model sampling matter for how you design tests for a Claude application?

**Q2.** What is the key difference between 'fast mode' and 'extended/adaptive thinking'?

**Q3.** Why should a production system re-run its eval suite before adopting a new model version?

**Q4.** What ordering requirement must be met for prompt caching to be effective?

**Q5.** For a high-volume, low-complexity workload, which model tier is generally the most cost-effective choice?

## Answers


**A1.** Because identical prompts can produce different outputs, tests should validate structure/correctness properties rather than expecting exact-text matches.

**A2.** Fast mode favors low latency for simple tasks; extended/adaptive thinking allocates more internal reasoning for harder problems at the cost of latency/tokens.

**A3.** Because new model releases can introduce breaking behavior changes that could silently degrade a carefully-tuned prompt or workflow.

**A4.** Static, reusable content (e.g., the system prompt) must be placed before dynamic, per-request content in the prompt structure.

**A5.** Haiku - it is optimized for fast, low-cost, high-volume, simple tasks.

---
**Next**: [Domain 6: Prompt and Context Engineering](./06-prompt-and-context-engineering.ipynb)
**Back**: [README](./README.md)